In [1]:
import numpy as np
from careamics import CAREamist
from careamics.config import create_n2v_configuration
from pathlib import Path
import tifffile
import matplotlib.pyplot as plt


In [2]:
train_path = Path("D:/Data/Spheroids-Data-OCProject/Individual_Images/zprojection")

In [8]:
config = create_n2v_configuration(
    experiment_name="n2v_spheroids",
    data_type="tiff",
    axes="TYX",
    patch_size=[64, 64],
    batch_size=1,
    num_epochs=10
)

print(config)

{'algorithm_config': {'algorithm': 'n2v',
                      'loss': 'n2v',
                      'lr_scheduler': {'name': 'ReduceLROnPlateau',
                                       'parameters': {}},
                      'model': {'architecture': 'UNet',
                                'conv_dims': 2,
                                'depth': 2,
                                'final_activation': 'None',
                                'in_channels': 1,
                                'independent_channels': True,
                                'n2v2': False,
                                'num_channels_init': 32,
                                'num_classes': 1},
                      'n2v_config': {'masked_pixel_percentage': 0.2,
                                     'name': 'N2VManipulate',
                                     'remove_center': True,
                                     'roi_size': 11,
                                     'strategy': 'uniform',
                

In [9]:
careamist = CAREamist(
    source=config,
    work_dir="../models/careamics"
    )

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [10]:
careamist.train(
    train_source=train_path
)

You are using a CUDA device ('NVIDIA GeForce RTX 4090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
Extracted 74382 patches from input array.
Computed dataset mean: [0.03588559], std: [0.06475669]
c:\Users\Caterina\anaconda3\envs\careamics\lib\site-packages\pytorch_lightning\callbacks\model_checkpoint.py:654: Checkpoint directory D:\GitHub\AI4Life-OC-3DM3\models\careamics\checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name  | Type | Params | Mode 
---------------------------------------
0 | model | UNet | 509 K  | train
---------------------------------------
509 K     Trainable params
0         Non-trainable params
509 K     Total params
2.037     Total estimated model params size (MB)
39 

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

c:\Users\Caterina\anaconda3\envs\careamics\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.
c:\Users\Caterina\anaconda3\envs\careamics\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.


Training: 0it [00:00, ?it/s]

Validating: 0it [00:00, ?it/s]

...

Validating: 0it [00:00, ?it/s]

Validating: 0it [00:00, ?it/s]

Validating: 0it [00:00, ?it/s]

Validating: 0it [00:00, ?it/s]

`Trainer.fit` stopped: `max_epochs=10` reached.


In [ ]:
prediction = careamist.predict(
    source=train_path,
    axes="TYX",
    tta=False
)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
c:\Users\Caterina\anaconda3\envs\careamics\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.


Predicting: |          | 0/? [00:00<?, ?it/s]

In [13]:
pred_folder = Path("D:/Data/Spheroids-Data-OCProject/Individual_Images/zprojection-denoised")
pred_folder.mkdir(exist_ok=True, parents=True)

final_data = np.concatenate(prediction)
tifffile.imwrite(pred_folder / "prediction.tiff", final_data)

In [19]:
import os
os.listdir(train_path)[0]

'l_2_2_20X-03-Scene-01_stitched.tif'

In [22]:
plot_image = tifffile.imread(os.listdir(train_path)[0])
# Show multiple slices
zs = [5, 10, 15, 20, 25, 30]

fig, ax = plt.subplots(len(zs), 2, figsize=(10, 5 * len(zs)))
for i, z in enumerate(zs):
    ax[i, 0].imshow(plot_image[z])
    ax[i, 0].set_title(f"Noisy - Plane {z}")

    ax[i, 1].imshow(prediction[0].squeeze()[z])
    ax[i, 1].set_title(f"Prediction - Plane {z}")

FileNotFoundError: [Errno 2] No such file or directory: 'D:\\GitHub\\AI4Life-OC-3DM3\\notebooks\\l_2_2_20X-03-Scene-01_stitched.tif'